In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import re

/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


In [3]:
df_amoli = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/Amoli TOKO 3C.csv")
df_berliancah = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/Berliancahh.csv")
df_crb = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/CRB phone second.csv")
df_galaxi = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/GALAXI. 29.csv")
df_amoli2 = pd.read_csv("data/HP SAMSUNG J2 PRIME NORMAL SIAP PAKAI SECOND BERKUALITAS.csv")
df_kka1 = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/K_KA SELULER (counterfeit).csv")
df_kka2 = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/K_KA SELULER (2).csv")
df_maja = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/maja stori (counterfeit).csv")
df_panda = pd.read_csv("/Users/asyzyni/Desktop/TA /Model-Try/data/Panda Elektronik Store.csv")

In [4]:
def parse_tanggal(teks_tanggal):
    """
    Mengubah teks tanggal menjadi format datetime.
    Menangani format 'YYYY-MM-DD' dan format relatif seperti '4 minggu lalu'.
    """
    if pd.isna(teks_tanggal):
        return pd.NaT
        
    teks_tanggal = str(teks_tanggal).strip().lower()
    
    # Jika formatnya standar YYYY-MM-DD
    if re.match(r'\d{4}-\d{2}-\d{2}', teks_tanggal):
        return pd.to_datetime(teks_tanggal, errors='coerce')
        
    # Jika formatnya relatif (misal: "4 minggu lalu", "3 hari lalu")
    # Asumsi waktu scraping adalah waktu saat ini
    waktu_sekarang = pd.Timestamp.now()
    
    try:
        angka = int(re.search(r'\d+', teks_tanggal).group())
        if 'hari' in teks_tanggal:
            return waktu_sekarang - timedelta(days=angka)
        elif 'minggu' in teks_tanggal:
            return waktu_sekarang - timedelta(weeks=angka)
        elif 'bulan' in teks_tanggal:
            # Aproksimasi 1 bulan = 30 hari
            return waktu_sekarang - timedelta(days=angka*30)
    except AttributeError:
        # Jika tidak ada angka yang ditemukan
        pass
        
    return pd.NaT 

def bersihkan_toko(df, nama_toko, kolom_ulasan, kolom_tanggal):
    """
    Mengekstrak kolom yang benar, membersihkan teks, dan menyamakan format.
    """
    df_clean = df.copy()
    
    # 1. Pastikan kolom yang diminta ada di dataframe
    if kolom_ulasan not in df_clean.columns or kolom_tanggal not in df_clean.columns:
        print(f"Peringatan: Kolom tidak ditemukan di data {nama_toko}. Melewati...")
        return pd.DataFrame()
        
    # 2. Ambil dan ubah nama kolom
    df_clean = df_clean[[kolom_ulasan, kolom_tanggal]].copy()
    df_clean.columns = ['ulasan', 'tanggal']
    df_clean['nama_toko'] = nama_toko
    
    # 3. Konversi format tanggal menggunakan fungsi bantuan
    df_clean['tanggal'] = df_clean['tanggal'].apply(parse_tanggal)
    
    # 4. Hapus baris yang ulasannya atau tanggalnya kosong/NaN
    df_clean = df_clean.dropna(subset=['ulasan', 'tanggal'])
    
    # 5. Pembersihan Teks Ulasan
    # Mengubah teks menjadi string
    df_clean['ulasan'] = df_clean['ulasan'].astype(str).str.strip()
    
    # Daftar tag anomali lazada yang sering ter-scrape sendirian
    tag_buang = ['harga:', 'daya tahan baterai:', 'tampilan:', 'performa:', 'kualitas:', 'kemudahan penggunaan:']
    
    # Hapus baris jika ulasannya HANYA berisi tag tersebut (case-insensitive)
    df_clean = df_clean[~df_clean['ulasan'].str.lower().isin(tag_buang)]
    
    # (Opsional) Jika teks dimulai dengan tag lalu diikuti ulasan (misal "kualitas: bagus banget")
    # Kita hapus tag-nya agar LLM nanti tidak bingung
    for tag in tag_buang:
        df_clean['ulasan'] = df_clean['ulasan'].apply(
            lambda x: re.sub(f'^{tag}\s*', '', x, flags=re.IGNORECASE)
        )
        
    # Hapus ulasan yang terlalu pendek (misal di bawah 4 karakter) atau cuma spasi
    df_clean = df_clean[df_clean['ulasan'].str.len() >= 4]
    
    # Susun ulang kolom
    return df_clean[['nama_toko', 'ulasan', 'tanggal']]


In [5]:
list_df_bersih = []
list_df_bersih.append(bersihkan_toko(df_amoli, 'Amoli', kolom_ulasan='data2', kolom_tanggal='phone'))
list_df_bersih.append(bersihkan_toko(df_berliancah, 'Berliancah', kolom_ulasan='data2', kolom_tanggal='phone'))
list_df_bersih.append(bersihkan_toko(df_panda, 'Panda', kolom_ulasan='data2', kolom_tanggal='date'))
list_df_bersih.append(bersihkan_toko(df_crb, 'CRB', kolom_ulasan='data2', kolom_tanggal='phone'))
# 1. Eksekusi df_galaxi
list_df_bersih.append(bersihkan_toko(df_galaxi, 'Galaxi', kolom_ulasan='data2', kolom_tanggal='phone'))

# 2. Eksekusi df_kka1 (Perhatikan: kolomnya data3 dan data2)
list_df_bersih.append(bersihkan_toko(df_kka1, 'KKA', kolom_ulasan='data3', kolom_tanggal='data2'))

# 3. Eksekusi df_kka2
list_df_bersih.append(bersihkan_toko(df_kka2, 'KKA', kolom_ulasan='data2', kolom_tanggal='phone'))

# 4. Eksekusi df_maja
list_df_bersih.append(bersihkan_toko(df_maja, 'Maja', kolom_ulasan='data2', kolom_tanggal='phone'))

In [6]:
# ==========================================
# GABUNGKAN SEMUANYA
# ==========================================
# Setelah semua dimasukkan ke list_df_bersih, baru di-concat
df_final_gabungan = pd.concat(list_df_bersih, ignore_index=True)

# Urutkan berdasarkan toko dan tanggal untuk kebutuhan HMM nanti
df_final_gabungan = df_final_gabungan.sort_values(by=['nama_toko', 'tanggal']).reset_index(drop=True)

# Tampilkan hasilnya
print(df_final_gabungan.head(15))

   nama_toko                                             ulasan    tanggal
0      Amoli  Thanks seller kualitas bagus sekali mantap san... 2023-12-13
1      Amoli      hp hebat dan baguss aku sangat senang bangett 2023-12-13
2      Amoli  Barangnya bagus, suka banget, warna sesuai pes... 2023-12-14
3      Amoli  Alhamdulillah barang sangat berkualitas sekali... 2023-12-14
4      Amoli  """Bagus hp ori 😍😍😍\nseller ramah bngt gercep.... 2023-12-15
5      Amoli  Karena keluaran terbaru hape layarnya amoled, ... 2023-12-15
6      Amoli  Hp murah barang berkualitas dan berfungsi. Pen... 2023-12-15
7      Amoli  Wihhh mantap juga nih toko, harga segitu bisa ... 2023-12-16
8      Amoli  Masih ga nyangka bisa beruntung dapetin hp ben... 2023-12-16
9      Amoli  Buat masukan penjual..!!!!untuk pengirimnaya t... 2023-12-20
10     Amoli  Barang sesuai dengan pesanan, baru dan masih s... 2023-12-20
11     Amoli  Alhamdulillah sampai dengan selamat, hehe🥰😍.\n... 2023-12-20
12     Amoli  Kualitas ok

In [7]:
df_final_gabungan['tanggal'] = pd.to_datetime(df_final_gabungan['tanggal'])

# Ekstrak bagian tanggalnya saja (tanpa jam, menit, detik)
df_final_gabungan['tanggal'] = df_final_gabungan['tanggal'].dt.date

In [8]:
df_final_gabungan

,nama_toko,ulasan,tanggal
0,Amoli,Thanks seller kualitas bagus sekali mantap san...,2023-12-13
1,Amoli,hp hebat dan baguss aku sangat senang bangett,2023-12-13
2,Amoli,"Barangnya bagus, suka banget, warna sesuai pes...",2023-12-14
3,Amoli,Alhamdulillah barang sangat berkualitas sekali...,2023-12-14
4,Amoli,"""""""Bagus hp ori 😍😍😍\nseller ramah bngt gercep....",2023-12-15
...,...,...,...
418,Panda,alhamdulillah hp nya udah datang pengiriman ce...,2026-05-10
419,Panda,kak kenapa gak ada sinyal nya,2026-05-12
420,Panda,"gimana ya kok gak ada sinyal, batrenya habis G...",2026-05-12
421,Panda,"Baru, segel, dan bergaransi, bagus n worth it",2026-05-15


In [9]:
df_final_gabungan.to_csv('data siap TA.csv', index=False)

In [16]:
df_model = df_final_gabungan.copy()

In [17]:
df_model = df_model.rename(columns={
    'nama_toko': 'shop_id', 
    'ulasan': 'review_text', 
    'tanggal': 'timestamp'
})

In [18]:
toko_counterfeit = ['KKA', 'Maja']
df_model['label'] = np.where(df_model['shop_id'].isin(toko_counterfeit), 1, 0)
df_model = df_model.sort_values(by=['shop_id', 'timestamp']).reset_index(drop=True)

In [21]:
df_model.to_csv('model data.csv', index=False)

In [24]:
df_model['label'].value_counts()

label
0    393
1     30
Name: count, dtype: int64

In [25]:
# Ambil hanya 30 data dari label 0 secara acak
df_label0 = df_model[df_model['label'] == 0].sample(n=30, random_state=42)

# Ambil semua data dari label 1 (jumlahnya 30)
df_label1 = df_model[df_model['label'] == 1]

# Gabungkan kembali
df_balanced = pd.concat([df_label0, df_label1])

# Acak ulang urutan baris
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

# Cek distribusi label
print(df_balanced['label'].value_counts())

label
0    30
1    30
Name: count, dtype: int64


In [27]:
df_balanced.to_csv("scraping_data_balanced.csv", index=False)